# 02 · Bronze con Auto Loader

**Objetivo:** que **solo los archivos nuevos** de RAW entren a la tabla bronze, sin duplicar nunca un archivo ni un vuelo.

    RAW (JSON por aeropuerto/tipo/día) ──► Auto Loader (cloudFiles) ──► MERGE por (icao24, firstSeen) ──► bronze_flights

Auto Loader corre **por lotes** (`trigger(availableNow=True)`): procesa lo pendiente y se detiene.

| Paso | Qué haces |
|---|---|
| 0 | Limpiar RAW de archivos de prueba |
| 1 | Crear bronze con Change Data Feed |
| 2 | LECTURA: Auto Loader detecta los archivos |
| 3 | ESCRITURA: MERGE lote a lote |
| 4 | ¿Qué archivos ya leyó Auto Loader? |
| 5 | Duplicados: un vuelo visto en dos aeropuertos |
| 6 | Re-ejecutar: no entra nada dos veces |
| 7 | `_rescued_data` |

In [0]:
%run ./00_Setup

## Paso 0 · Limpiar RAW y empezar de cero
Se borran los archivos de prueba (marzo 2025 y Lima) **antes** de que Auto Loader los lea.
`reset_pipeline()` borra tablas y la memoria de Auto Loader; **RAW se conserva**.

In [0]:
borrar = [f for f in raw_files() if "_2025-" in f or f.startswith("flights_SPJC_")]
for f in borrar:
    dbutils.fs.rm(f"{RAW_PATH}/{f}")
print(f"Borrados {len(borrar)} archivos de prueba")

reset_pipeline()
print("Archivos en RAW:", len(raw_files()))

## Paso 1 · Crear bronze con Change Data Feed activado
La tabla se crea **antes** de la primera carga con `delta.enableChangeDataFeed = true`:
así cada cambio queda registrado desde la versión 0 y silver podrá leerlos.

In [0]:
create_bronze_table()
display(spark.sql(f"SHOW TBLPROPERTIES {BRONZE}").where("key = 'delta.enableChangeDataFeed'"))

## Paso 2 · LECTURA · Auto Loader detecta los archivos nuevos
`cloudFiles` es la fuente de Structured Streaming que vigila una carpeta. Es el mismo código de `autoloader_read()`.

In [0]:
lectura = (spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("multiLine", "true")                              # cada archivo es un arreglo JSON
    .option("cloudFiles.schemaLocation", SCHEMA_PATH)         # dónde guarda el esquema
    .option("cloudFiles.rescuedDataColumn", "_rescued_data")  # lo que no encaje en el esquema va aquí
    .schema(ESQUEMA_VUELOS)                                   # esquema explícito
    .load(RAW_PATH)
    .select("*",
            F.col("_metadata.file_name").alias("source_file"),   # de qué archivo viene cada vuelo
            F.current_timestamp().alias("ingested_at")))

print("¿Es un stream?", lectura.isStreaming)
lectura.printSchema()

👀 **Qué observar:** todavía no se leyó nada: un stream es un plan. Fíjate en las columnas extra:
`_rescued_data`, `source_file` e `ingested_at`.

## Paso 3 · ESCRITURA · cada lote entra a bronze con `MERGE`
- `foreachBatch(upsert_to_bronze)`: cada lote hace `MERGE` por `(icao24, firstSeen)`. Si el vuelo ya existe, se **actualiza**; si no, se **inserta**.
- `checkpointLocation`: la **memoria** de Auto Loader (qué archivos ya procesó).
- `trigger(availableNow=True)`: procesa todo lo pendiente y se detiene.

In [0]:
query = (lectura.writeStream
    .foreachBatch(upsert_to_bronze)
    .option("checkpointLocation", CHECKPOINT_PATH)
    .trigger(availableNow=True)
    .start())

query.awaitTermination()
print("Vuelos en bronze:", spark.table(BRONZE).count())

In [0]:
show_history(BRONZE)

👀 **Qué observar:** la versión 0 es el `CREATE TABLE` y luego aparece un `MERGE`.
En `operationMetrics` mira `numTargetRowsInserted` y `numTargetRowsUpdated`.

## Paso 4 · ¿Qué archivos ya leyó Auto Loader?

In [0]:
display(spark.sql(f"""
    SELECT path, size, discovery_time, commit_time
    FROM cloud_files_state('{CHECKPOINT_PATH}')
    ORDER BY path
"""))

👀 **Qué observar:** un registro por archivo de RAW. Esta lista es la que impide que un archivo entre dos veces.

## Paso 5 · Duplicados: un vuelo visto en dos aeropuertos
Un vuelo São Paulo → Río aparece en `SBGR_departure` **y** en `SBGL_arrival`.
Comparamos cuántos registros hay en RAW contra cuántos vuelos únicos quedaron en bronze.

In [0]:
raw_df = spark.read.option("multiLine", "true").schema(ESQUEMA_VUELOS).json(RAW_PATH)

total_raw = raw_df.count()
unicos    = spark.table(BRONZE).count()
print(f"Registros en RAW    : {total_raw}")
print(f"Vuelos en bronze    : {unicos}")
print(f"Duplicados resueltos: {total_raw - unicos}")

In [0]:
# Ejemplos: vuelos que aparecen en más de un archivo de RAW
ejemplos = (raw_df.withColumn("archivo", F.col("_metadata.file_name"))
            .groupBy("icao24", "firstSeen", F.trim("callsign").alias("callsign"),
                     "estDepartureAirport", "estArrivalAirport")
            .agg(F.count("*").alias("veces"), F.collect_list("archivo").alias("archivos"))
            .where("veces > 1")
            .orderBy(F.col("firstSeen").desc()))
display(ejemplos.limit(10))

In [0]:
# Uno de esos vuelos en bronze: una sola fila
ej = ejemplos.first()
display(spark.table(BRONZE)
        .where((F.col("icao24") == ej["icao24"]) & (F.col("firstSeen") == ej["firstSeen"]))
        .select("icao24", "callsign", "estDepartureAirport", "estArrivalAirport", "source_file"))

👀 **Qué observar:** en RAW el vuelo está en dos archivos (salida y llegada); en bronze hay **una sola fila**,
porque el `MERGE` por `(icao24, firstSeen)` lo reconoce como el mismo vuelo.

## Paso 6 · Re-ejecutar: no entra nada dos veces
`run_bronze()` hace lectura + escritura con las mismas funciones del setup.

In [0]:
version_antes = table_version(BRONZE)
run_bronze()
print("Versión antes:", version_antes, "| después:", table_version(BRONZE))

👀 **Qué observar:** "sin archivos nuevos" y la versión no cambia: no hubo ningún commit.

🧪 **Cuando recuperes créditos de OpenSky:** corre el notebook 01 para completar los días faltantes y luego `run_bronze()`.
En `cloud_files_state` verás que entran **solo** los archivos nuevos.

## Paso 7 · `_rescued_data`: lo que no estaba en el esquema no se pierde
El esquema explícito solo declara los campos que usamos; el resto (distancias al aeropuerto, etc.) queda en `_rescued_data`.

In [0]:
display(spark.table(BRONZE)
        .select("icao24", "callsign",
                F.get_json_object("_rescued_data", "$.estDepartureAirportHorizDistance").alias("dist_horiz_salida"),
                "_rescued_data")
        .limit(5))

## ✔ Checkpoint
- [ ] Bronze tiene CDF activado
- [ ] `cloud_files_state` muestra un registro por archivo de RAW
- [ ] Vi cuántos duplicados resolvió el `MERGE` (registros RAW vs vuelos en bronze)
- [ ] Re-ejecutar `run_bronze()` no cambia la versión
- [ ] Entiendo para qué sirve `_rescued_data`

**Siguiente:** `03_Silver_con_CDF`